In [28]:
import numpy as np
import pandas as pd
import wave
import math
import struct
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Activation, Input

In [3]:
# Prepare a dummy data for simulating musical notes
notes_freqs = {
    'A': 440.0, 'B': 493.88, 'C': 261.63, 'D': 293.66, 'E': 393.63,
    'F': 349.23, 'G': 392.0
}

notes_freqs ;

In [4]:
notes = list(notes_freqs.keys())
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [5]:
note_to_int = {note: i for i , note in enumerate(notes)}

In [6]:
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [7]:
int_to_note = {i: note for i, note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [18]:
raw_music_data = [notes[np.random.randint(0,7)]for i in range(1000)]

#### Data Preparation

In [19]:
seq_length = 3
network_input = []
network_output = []

for i in range (len(raw_music_data) - seq_length):
    seq_in = raw_music_data[i : i+ seq_length]
    seq_out =raw_music_data[i + seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in, '-->', seq_out)

['G', 'A', 'D'] --> D
['A', 'D', 'D'] --> C
['D', 'D', 'C'] --> F
['D', 'C', 'F'] --> C
['C', 'F', 'C'] --> A
['F', 'C', 'A'] --> A
['C', 'A', 'A'] --> B
['A', 'A', 'B'] --> E
['A', 'B', 'E'] --> F
['B', 'E', 'F'] --> C
['E', 'F', 'C'] --> F
['F', 'C', 'F'] --> A
['C', 'F', 'A'] --> C
['F', 'A', 'C'] --> G
['A', 'C', 'G'] --> C
['C', 'G', 'C'] --> C
['G', 'C', 'C'] --> F
['C', 'C', 'F'] --> A
['C', 'F', 'A'] --> E
['F', 'A', 'E'] --> D
['A', 'E', 'D'] --> C
['E', 'D', 'C'] --> D
['D', 'C', 'D'] --> D
['C', 'D', 'D'] --> F
['D', 'D', 'F'] --> G
['D', 'F', 'G'] --> D
['F', 'G', 'D'] --> G
['G', 'D', 'G'] --> A
['D', 'G', 'A'] --> B
['G', 'A', 'B'] --> F
['A', 'B', 'F'] --> B
['B', 'F', 'B'] --> E
['F', 'B', 'E'] --> B
['B', 'E', 'B'] --> C
['E', 'B', 'C'] --> B
['B', 'C', 'B'] --> D
['C', 'B', 'D'] --> B
['B', 'D', 'B'] --> F
['D', 'B', 'F'] --> C
['B', 'F', 'C'] --> A
['F', 'C', 'A'] --> B
['C', 'A', 'B'] --> G
['A', 'B', 'G'] --> E
['B', 'G', 'E'] --> D
['G', 'E', 'D'] --> C
['E', 'D',

In [20]:
n_patterns = len(network_input)

n_patterns

x = np.reshape(network_input, (n_patterns, seq_length, 1))
x

In [24]:
from keras.utils import to_categorical

In [25]:
y = to_categorical(network_output)

In [26]:
y.shape

(997, 7)

In [27]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]])

#### Build the model

In [29]:
model = Sequential()
model.add(Input((3,1)))
model.add(GRU(256))
model.add(Dense(512, activation = 'relu'))
model.add(Dense(7, activation = 'softmax'))

In [30]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 256)            │       198,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,087 (1.27 MB)

 Trainable params: 334,087 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [33]:
model.fit(x, y, epochs = 100, batch_size = 10)

Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1464 - loss: 1.9667
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1384 - loss: 1.9523
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1525 - loss: 1.9500
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1525 - loss: 1.9431
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1505 - loss: 1.9415
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1665 - loss: 1.9410
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1715 - loss: 1.9394
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1635 - loss: 1.9338
Epoch 9/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1565 - loss: 1.9338
Epoch 10/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1695 - loss: 1.9287
Epoch 11/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1735 - loss: 1.9256
Epoch 12/100
100/100 ━━━━━━━━━━━━━━━━━━━━

In [34]:
model.fit(x, y, epochs = 1000, batch_size = 10)

Epoch 1/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3290 - loss: 1.5903
Epoch 2/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3220 - loss: 1.5860
Epoch 3/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3280 - loss: 1.5803
Epoch 4/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3270 - loss: 1.5852
Epoch 5/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3230 - loss: 1.5801
Epoch 6/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3260 - loss: 1.5867
Epoch 7/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3350 - loss: 1.5719
Epoch 8/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3380 - loss: 1.5653
Epoch 9/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3340 - loss: 1.5603
Epoch 10/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3390 - loss: 1.5647
Epoch 11/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3300 - loss: 1.5592
Epoch 12/1000
100/100 ━━━━━━━━

In [35]:
#### Genrate new melody sequence 

In [39]:
start_index = np.random.randint(0, len(network_output))
pattern = network_input[start_index]
pattern = network_input[start_index]
pattern

[6, 6, 3]

In [47]:
generated_melody = []
for i in range(16):
    x_input = np.reshape(pattern, (1, len(pattern), 1))
    pred = model.predict(x_input, verbose = False)
    index = np.argmax(pred)
    result = int_to_note[index]
    generated_melody.append(result)
    pattern.append(index)
    pattern = pattern[1:len(pattern)]

In [48]:
generated_melody

['C',
 'A',
 'G',
 'C',
 'A',
 'G',
 'C',
 'A',
 'G',
 'C',
 'A',
 'G',
 'C',
 'A',
 'G',
 'C']

#### Save this as audio file

In [49]:
with wave.open('my_music.wav','w') as wav_file:
    wav_file.setparams((1, 2,44100,0,'NONE' , 'not compressed'))
    for note in generated_melody:
        freq = notes_freqs[note]
        num_samples = int(0.5 * 44100)
        for i in range(num_samples):
            t = float(i)/44100
            value = int(32767 * 0.5 * math.sin(2*math.pi*freq*t))
            data = struct.pack('<h', value)
            wav_file.writeframes(data)